# 🔧 MI Evaluation Fix
## Correcting Membership Inference Measurement for Truth Serum

---

**Problem:** Standard MI assumes low loss = member. But Truth Serum makes targets have HIGH loss (due to conflicting labels). We need to flip the evaluation.

**Fix:** Use high loss as the attack signal for targets.

---

⏱️ **Runtime:** ~20-30 min (retrain 3 models + evaluate)

In [ ]:
#@title 1. Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), 'Need GPU!'
device = torch.device('cuda')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

EPOCHS = 25
N_TARGETS = 250
BATCH_SIZE = 256

In [ ]:
#@title 2. Data + Classes
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_features = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainset_features = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_features)

class TruthSerumAttack:
    def __init__(self, dataset, n_targets=250, n_copies=8, seed=42):
        self.dataset = dataset
        self.n_targets = n_targets
        self.n_copies = n_copies
        np.random.seed(seed)
        self.target_indices = np.random.choice(len(dataset), n_targets, replace=False)
        
    def create_poisoned_dataset(self):
        images, labels, is_poison, original_indices = [], [], [], []
        for i in range(len(self.dataset)):
            img, label = self.dataset[i]
            images.append(img)
            labels.append(label)
            is_poison.append(False)
            original_indices.append(i)
        
        for target_idx in self.target_indices:
            img, true_label = self.dataset[target_idx]
            wrong_label = (true_label + np.random.randint(1, 10)) % 10
            for _ in range(self.n_copies):
                images.append(img.clone() if hasattr(img, 'clone') else img)
                labels.append(wrong_label)
                is_poison.append(True)
                original_indices.append(target_idx)
        return PoisonedDataset(images, labels, is_poison, original_indices)

class PoisonedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, is_poison, original_indices):
        self.images = images
        self.labels = labels
        self.is_poison = is_poison
        self.original_indices = original_indices
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]
    def get_poison_mask(self): return np.array(self.is_poison)

def create_resnet18():
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)

def train_model(model, trainloader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.train()
    for epoch in range(epochs):
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1}/{epochs}')
    return model

def evaluate_accuracy(model, testloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            _, predicted = model(inputs).max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return correct / total

print('✅ Setup complete')

In [ ]:
#@title 3. CORRECTED MI Attack Functions

def get_losses(model, loader):
    """Get per-sample losses."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')
    losses = []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            losses.extend(loss.cpu().numpy())
    return np.array(losses)


def compute_mi_truth_serum(model, target_loader, nonmember_loader):
    """
    CORRECTED MI attack for Truth Serum.
    
    Truth Serum mechanism:
    - Targets have HIGH loss (model learned wrong label from poison copies)
    - Non-members have normal/lower loss
    
    So: HIGH loss = likely poisoned target = positive class
    """
    target_losses = get_losses(model, target_loader)
    nonmember_losses = get_losses(model, nonmember_loader)
    
    # CORRECT: Use loss directly (high loss = likely target)
    # NOT: -loss (which assumes low loss = member)
    scores = np.concatenate([target_losses, nonmember_losses])
    labels = np.concatenate([np.ones(len(target_losses)), np.zeros(len(nonmember_losses))])
    
    auc = roc_auc_score(labels, scores)
    
    return auc, target_losses, nonmember_losses


def compute_mi_standard(model, member_loader, nonmember_loader):
    """
    Standard MI attack (for clean model baseline).
    Low loss = likely member.
    """
    member_losses = get_losses(model, member_loader)
    nonmember_losses = get_losses(model, nonmember_loader)
    
    # Standard: low loss = member, so use -loss as score
    scores = np.concatenate([-member_losses, -nonmember_losses])
    labels = np.concatenate([np.ones(len(member_losses)), np.zeros(len(nonmember_losses))])
    
    return roc_auc_score(labels, scores)

print('✅ Corrected MI functions defined')

In [ ]:
#@title 4. Antidote Defense
class AntidoteDefense:
    def __init__(self, similarity_threshold=0.99):
        self.similarity_threshold = similarity_threshold
        self.feature_extractor = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.feature_extractor.fc = nn.Identity()
        self.feature_extractor.eval().to(device)
        
    def extract_features(self, dataset):
        loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False, num_workers=2)
        features, labels = [], []
        with torch.no_grad():
            for imgs, lbls in tqdm(loader, desc='Extracting'):
                features.append(self.feature_extractor(imgs.to(device)).cpu().numpy())
                labels.append(lbls.numpy())
        return np.vstack(features), np.concatenate(labels)
    
    def detect(self, dataset_labels, features):
        n = len(features)
        norms = np.linalg.norm(features, axis=1, keepdims=True)
        features_norm = features / (norms + 1e-8)
        suspected = np.zeros(n, dtype=bool)
        
        batch_size = 2000
        for i in tqdm(range(0, n, batch_size), desc='Detecting'):
            end_i = min(i + batch_size, n)
            sims = features_norm[i:end_i] @ features_norm.T
            
            for j, row_idx in enumerate(range(i, end_i)):
                row_sims = sims[j].copy()
                row_sims[row_idx] = 0
                duplicates = np.where(row_sims > self.similarity_threshold)[0]
                if len(duplicates) > 0:
                    if np.any(dataset_labels[duplicates] != dataset_labels[row_idx]):
                        suspected[row_idx] = True
                    if len(duplicates) >= 3:
                        suspected[row_idx] = True
        return suspected

print('✅ Defense defined')

In [ ]:
#@title 5. Create Datasets
print('='*60)
print('CREATING DATASETS')
print('='*60)

# Attack
attack = TruthSerumAttack(trainset, n_targets=N_TARGETS, n_copies=8, seed=42)
poisoned_dataset = attack.create_poisoned_dataset()
gt = poisoned_dataset.get_poison_mask()

# Feature version for detection
attack_feat = TruthSerumAttack(trainset_features, n_targets=N_TARGETS, n_copies=8, seed=42)
poisoned_feat = attack_feat.create_poisoned_dataset()

print(f'Total: {len(poisoned_dataset)}, Poisons: {gt.sum()}')

# Detection
print('\nRunning Antidote detection...')
defense = AntidoteDefense(0.99)
features, _ = defense.extract_features(poisoned_feat)
labels_arr = np.array([poisoned_dataset[i][1] for i in range(len(poisoned_dataset))])
suspected = defense.detect(labels_arr, features)

# Create defended dataset
keep = ~suspected
defended_dataset = PoisonedDataset(
    [poisoned_dataset.images[i] for i in np.where(keep)[0]],
    [poisoned_dataset.labels[i] for i in np.where(keep)[0]],
    [poisoned_dataset.is_poison[i] for i in np.where(keep)[0]],
    [poisoned_dataset.original_indices[i] for i in np.where(keep)[0]]
)
print(f'Removed {suspected.sum()}, kept {len(defended_dataset)}')

In [ ]:
#@title 6. Train Models
print('='*60)
print('TRAINING MODELS')
print('='*60)

testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

# Clean
print('\n1. CLEAN model...')
clean_loader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
clean_model = train_model(create_resnet18(), clean_loader)
clean_acc = evaluate_accuracy(clean_model, testloader)
print(f'   Accuracy: {clean_acc*100:.1f}%')

# Poisoned
print('\n2. POISONED model...')
poisoned_loader = torch.utils.data.DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
poisoned_model = train_model(create_resnet18(), poisoned_loader)
poisoned_acc = evaluate_accuracy(poisoned_model, testloader)
print(f'   Accuracy: {poisoned_acc*100:.1f}%')

# Defended
print('\n3. DEFENDED model...')
defended_loader = torch.utils.data.DataLoader(defended_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
defended_model = train_model(create_resnet18(), defended_loader)
defended_acc = evaluate_accuracy(defended_model, testloader)
print(f'   Accuracy: {defended_acc*100:.1f}%')

In [ ]:
#@title 7. CORRECTED MI Evaluation
print('='*60)
print('CORRECTED MEMBERSHIP INFERENCE EVALUATION')
print('='*60)

# Create target loader (the 250 targets with TRUE labels)
target_data = [(trainset[i][0], trainset[i][1]) for i in attack.target_indices]
target_dataset = torch.utils.data.TensorDataset(
    torch.stack([x[0] for x in target_data]),
    torch.tensor([x[1] for x in target_data])
)
target_loader = torch.utils.data.DataLoader(target_dataset, batch_size=256)

# Non-members = test set (never seen during training)
nonmember_loader = testloader

print('\nEvaluating MI attack on TARGETS (using corrected high-loss signal)...\n')

# Corrected evaluation
clean_auc, clean_target_losses, clean_nonmember_losses = compute_mi_truth_serum(
    clean_model, target_loader, nonmember_loader)
poisoned_auc, poisoned_target_losses, poisoned_nonmember_losses = compute_mi_truth_serum(
    poisoned_model, target_loader, nonmember_loader)
defended_auc, defended_target_losses, defended_nonmember_losses = compute_mi_truth_serum(
    defended_model, target_loader, nonmember_loader)

print(f'MI Attack AUC (high loss = target):')
print(f'  Clean model:    {clean_auc:.3f}')
print(f'  Poisoned model: {poisoned_auc:.3f}')
print(f'  Defended model: {defended_auc:.3f}')

# Calculate effectiveness
attack_amplification = poisoned_auc - clean_auc
defense_reduction = poisoned_auc - defended_auc
effectiveness = (defense_reduction / attack_amplification) * 100 if attack_amplification > 0 else 0

print(f'\n--- Analysis ---')
print(f'Attack amplification: {attack_amplification:.3f} ({clean_auc:.3f} → {poisoned_auc:.3f})')
print(f'Defense reduction:    {defense_reduction:.3f} ({poisoned_auc:.3f} → {defended_auc:.3f})')
print(f'Defense effectiveness: {effectiveness:.1f}%')

# Loss distribution analysis
print(f'\n--- Loss Distributions ---')
print(f'Clean model    - Target mean: {clean_target_losses.mean():.3f}, Non-member mean: {clean_nonmember_losses.mean():.3f}')
print(f'Poisoned model - Target mean: {poisoned_target_losses.mean():.3f}, Non-member mean: {poisoned_nonmember_losses.mean():.3f}')
print(f'Defended model - Target mean: {defended_target_losses.mean():.3f}, Non-member mean: {defended_nonmember_losses.mean():.3f}')

In [ ]:
#@title 8. Generate Corrected Figure
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Loss distributions
ax = axes[0]
ax.hist(clean_target_losses, bins=50, alpha=0.5, label='Clean - Targets', color='#2a9d8f', density=True)
ax.hist(clean_nonmember_losses, bins=50, alpha=0.5, label='Clean - Non-members', color='#264653', density=True)
ax.hist(poisoned_target_losses, bins=50, alpha=0.5, label='Poisoned - Targets', color='#e63946', density=True)
ax.set_xlabel('Loss')
ax.set_ylabel('Density')
ax.set_title('(a) Loss Distributions')
ax.legend(fontsize=8)

# Plot 2: MI Attack AUC Comparison
ax = axes[1]
models = ['Clean', 'Poisoned', 'Defended']
aucs = [clean_auc, poisoned_auc, defended_auc]
colors = ['#2a9d8f', '#e63946', '#f4a261']
bars = ax.bar(models, aucs, color=colors, edgecolor='black', linewidth=1.5)
ax.axhline(y=0.5, color='gray', linestyle='--', label='Random (AUC=0.5)')
ax.set_ylabel('MI Attack AUC')
ax.set_title('(b) Membership Inference Attack Success')
ax.set_ylim(0, 1.0)
ax.legend()
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

# Plot 3: ROC Curves
ax = axes[2]
for model, losses, nm_losses, name, color in [
    (clean_model, clean_target_losses, clean_nonmember_losses, 'Clean', '#2a9d8f'),
    (poisoned_model, poisoned_target_losses, poisoned_nonmember_losses, 'Poisoned', '#e63946'),
    (defended_model, defended_target_losses, defended_nonmember_losses, 'Defended', '#f4a261'),
]:
    scores = np.concatenate([losses, nm_losses])
    labels = np.concatenate([np.ones(len(losses)), np.zeros(len(nm_losses))])
    fpr, tpr, _ = roc_curve(labels, scores)
    ax.plot(fpr, tpr, label=name, color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('(c) ROC Curves')
ax.legend()

plt.tight_layout()
plt.savefig('mi_corrected.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Saved: mi_corrected.png')

In [ ]:
#@title 9. Update Results JSON
# Load previous results
try:
    with open('ablation_results.json', 'r') as f:
        results = json.load(f)
except:
    results = {}

# Update with corrected MI values
results['end_to_end'] = {
    'clean_acc': clean_acc,
    'poisoned_acc': poisoned_acc,
    'defended_acc': defended_acc,
    'clean_mi_auc': float(clean_auc),
    'poisoned_mi_auc': float(poisoned_auc),
    'defended_mi_auc': float(defended_auc),
    'attack_amplification': float(attack_amplification),
    'defense_reduction': float(defense_reduction),
    'effectiveness_pct': float(effectiveness),
    'target_loss_clean': float(clean_target_losses.mean()),
    'target_loss_poisoned': float(poisoned_target_losses.mean()),
    'target_loss_defended': float(defended_target_losses.mean()),
}

with open('ablation_results_corrected.json', 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Saved: ablation_results_corrected.json')
print('\n--- Corrected End-to-End Results ---')
print(json.dumps(results['end_to_end'], indent=2))

In [ ]:
#@title 10. Summary for Report
print('='*70)
print('CORRECTED RESULTS FOR FINAL REPORT')
print('='*70)

print('\n### Table: End-to-End Defense Effectiveness')
print(f'{"Model":<15} {"Test Acc":<12} {"MI AUC":<12} {"Target Loss":<12}')
print('-'*51)
print(f'{"Clean":<15} {clean_acc*100:<12.1f} {clean_auc:<12.3f} {clean_target_losses.mean():<12.3f}')
print(f'{"Poisoned":<15} {poisoned_acc*100:<12.1f} {poisoned_auc:<12.3f} {poisoned_target_losses.mean():<12.3f}')
print(f'{"Defended":<15} {defended_acc*100:<12.1f} {defended_auc:<12.3f} {defended_target_losses.mean():<12.3f}')

print(f'\n### Key Metrics')
print(f'- Attack amplification: +{attack_amplification:.3f} AUC (Truth Serum boosts MI attack)')
print(f'- Defense reduction: -{defense_reduction:.3f} AUC (Antidote neutralizes attack)')
print(f'- Defense effectiveness: {effectiveness:.1f}% of attack neutralized')
print(f'- Accuracy cost: {(clean_acc - defended_acc)*100:.1f}% drop from clean baseline')

print(f'\n### Interpretation')
if poisoned_auc > 0.9:
    print('- Poisoned model: SEVERE privacy vulnerability (AUC > 0.9)')
elif poisoned_auc > 0.7:
    print('- Poisoned model: SIGNIFICANT privacy vulnerability (AUC > 0.7)')
else:
    print('- Poisoned model: MODERATE privacy vulnerability')

if effectiveness > 80:
    print('- Defense: HIGHLY EFFECTIVE (>80% neutralization)')
elif effectiveness > 50:
    print('- Defense: EFFECTIVE (>50% neutralization)')
else:
    print('- Defense: PARTIALLY EFFECTIVE')

In [ ]:
#@title 11. Download
from google.colab import files
files.download('mi_corrected.png')
files.download('ablation_results_corrected.json')
print('✅ Done!')